List of stocks and returns historical construct an optimal portfolio.
Given that the return (Assume a return) advice what is the lowest risk portfolio.
Mean variance portfolio
Normally distributed portfolio
Given the risk tolerance what is the best return ETFs


Markowitz package


In [2]:
!pip install empyrical

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for empyrical: filename=empyrical-0.5.5-py3-none-any.whl size=39753 sha256=b2d0825741a547b5e282b143347a8e309d0ce0fe5e6b7ce311661465b947d1f8
  Stored in directory: /root/.cache/pip/wheels/ac/1d/58/a7ae5ef5c8de7c4b769f24c2584f4706564921f031b16b9cb6
Successfully built empyrical


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from empyrical import max_drawdown

# Load ETF Data
def load_etf_data(files):
    data = {}
    for etf, file in files.items():
        df = pd.read_csv(file, parse_dates=['Date'])
        df.set_index('Date', inplace=True)
        df.rename(columns={'Return': etf}, inplace=True)
        df[etf] = df[etf] / 100  # Convert percentage returns to decimal
        data[etf] = df[etf]
    return pd.DataFrame(data).dropna()

# Compute portfolio statistics
def portfolio_statistics(weights, mean_returns, cov_matrix):
    portfolio_return = np.dot(weights, mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return portfolio_return, portfolio_volatility

# Objective function for minimizing volatility
def min_volatility(weights, mean_returns, cov_matrix):
    return portfolio_statistics(weights, mean_returns, cov_matrix)[1]

# Find optimal portfolio using Markowitz optimization without minimum weight constraints
def optimal_portfolio(returns_df):
    mean_returns = returns_df.mean()
    cov_matrix = returns_df.cov()
    num_assets = len(mean_returns)

    initial_weights = np.ones(num_assets) / num_assets
    bounds = tuple((0, 1) for _ in range(num_assets))  # Allow weights to be 0 (no minimum allocation)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

    result = minimize(min_volatility, initial_weights, args=(mean_returns, cov_matrix),
                      method='SLSQP', bounds=bounds, constraints=constraints)
    weights = np.array(result.x) / np.sum(result.x)  # Normalize weights
    return weights

# Compute annual statistics
def compute_annual_statistics(df, strategy_returns, initial_cash=100):
    annual_stats = []
    cash = initial_cash
    for year in range(2000, 2025):
        if year not in df.index.year:
            continue
        annual_return = (1 + strategy_returns.loc[str(year)]).prod() - 1  # Compounded annual return
        cash *= (1 + annual_return)  # Convert to cash value
        annual_volatility = strategy_returns.loc[str(year)].std() * np.sqrt(252)
        sharpe_ratio = annual_return / annual_volatility if annual_volatility > 0 else np.nan
        mdd = max_drawdown(strategy_returns.loc[str(year)])
        annual_stats.append([year, cash, annual_return, annual_volatility, sharpe_ratio, mdd])
    return pd.DataFrame(annual_stats, columns=['Year', 'Cash Value', 'Annual Return', 'Volatility', 'Sharpe Ratio', 'Max Drawdown'])

# Load Data
etf_files = {
    'xlb': "XLB_weekly_return_detailed.csv",
    'xle': "XLE_weekly_return_detailed.csv",
    'xlf': "XLF_weekly_return_detailed.csv",
    'xli': "XLI_weekly_return_detailed.csv",
    'xlk': "XLK_weekly_return_detailed.csv",
    'xlp': "XLP_weekly_return_detailed.csv",
    'xlu': "XLU_weekly_return_detailed.csv",
    'xlv': "XLV_weekly_return_detailed.csv",
    'xly': "XLY_weekly_return_detailed.csv"
}
daily_returns = load_etf_data(etf_files)

# Compute Markowitz Strategy Returns
opt_weights = optimal_portfolio(daily_returns)
markowitz_returns = daily_returns.dot(opt_weights)  # Properly apply weights

# Display Weights Assigned to Each ETF
weights_df = pd.DataFrame({'ETF': daily_returns.columns, 'Weight': opt_weights})
print("Optimal Portfolio Weights:")
print(weights_df)

# Compute Annual Statistics in Cash
markowitz_stats = compute_annual_statistics(daily_returns, markowitz_returns)

# Save Results to CSV
markowitz_stats.to_csv("markowitz_annual_cash.csv", index=False)

# Display the final comparison DataFrame
print("Markowitz Annual Returns in Cash:")
print(markowitz_stats)


Optimal Portfolio Weights:
   ETF        Weight
0  xlb  6.625128e-18
1  xle  0.000000e+00
2  xlf  0.000000e+00
3  xli  7.329042e-18
4  xlk  1.092343e-17
5  xlp  5.849627e-01
6  xlu  1.385162e-01
7  xlv  2.765211e-01
8  xly  0.000000e+00
Markowitz Annual Returns in Cash:
    Year  Cash Value  Annual Return  Volatility  Sharpe Ratio  Max Drawdown
0   2000  115.127601       0.151276    0.183918      0.822519     -0.181155
1   2001  107.023912      -0.070389    0.152785     -0.460706     -0.160807
2   2002   90.103357      -0.158101    0.196031     -0.806509     -0.286287
3   2003  103.193456       0.145279    0.129055      1.125708     -0.118779
4   2004  111.572234       0.081195    0.091441      0.887947     -0.075577
5   2005  117.993887       0.057556    0.088771      0.648363     -0.047024
6   2006  133.744404       0.133486    0.080635      1.655441     -0.048114
7   2007  149.898011       0.120780    0.109968      1.098322     -0.064807
8   2008  121.663335      -0.188359    0.2517

In [5]:
print("Markowitz Annual Returns in Cash:")
print(markowitz_stats.describe())  # Summary statistics

Markowitz Annual Returns in Cash:
              Year  Cash Value  Annual Return  Volatility  Sharpe Ratio  \
count    25.000000   25.000000      25.000000   25.000000     25.000000   
mean   2012.000000  297.382257       0.090915    0.137831      0.949311   
std       7.359801  209.797296       0.123491    0.059664      1.050397   
min    2000.000000   85.387111      -0.236821    0.056720     -0.840080   
25%    2006.000000  121.551355       0.031191    0.094997      0.242400   
50%    2012.000000  196.536828       0.109845    0.121998      0.790492   
75%    2018.000000  392.994979       0.168079    0.168863      1.483214   
max    2024.000000  743.471400       0.279718    0.305411      2.963336   

       Max Drawdown  
count     25.000000  
mean      -0.120521  
std        0.087068  
min       -0.329338  
25%       -0.159020  
50%       -0.094669  
75%       -0.061663  
max       -0.025889  
